# 04 · 网关路由、Fallback 与可重放边界

目标：调用仓库中的真实 Router，并理解为什么流式响应开始后不能盲目 fallback。对应文档：`04_real_projects/4.2-gateway-and-proxy.md`。

## 从前端请求重试迁移

浏览器里重试 fetch 常被看作重新调用一次；网关还要判断请求体能否重放、响应是否已经开始、错误是否可重试，以及总 deadline 是否允许下一次尝试。

In [ ]:
!*cd "$GONB_TMP_DIR" && go mod edit -require=example.com/architecture-labs@v0.0.0 -replace=example.com/architecture-labs="$GO_LEARNING_LABS"

In [ ]:
import (
    "errors"
    "fmt"

    gatewaylab "example.com/architecture-labs/gateway"
)

## 实验一：健康路由

首选 provider 不健康时，Router 会选择第一个健康候选；全部不健康时返回稳定的 `ErrNoHealthyProvider`，供上层映射为降级响应。

In [ ]:
%%
router := gatewaylab.NewRouter([]gatewaylab.Provider{
    {Name: "primary", Healthy: false, Weight: 100},
    {Name: "backup", Healthy: true, Weight: 10},
})

selected, selectErr := router.Select("primary")
fmt.Printf("preferred=primary selected=%s err=%v\n", selected.Name, selectErr)

unavailable := gatewaylab.NewRouter([]gatewaylab.Provider{{Name: "primary", Healthy: false}})
_, unavailableErr := unavailable.Select("primary")
fmt.Println("稳定错误分类:", errors.Is(unavailableErr, gatewaylab.ErrNoHealthyProvider))

In [ ]:
type ProviderAttempt struct {
    Provider string
    Started  bool
    Err      error
}

func chooseFallback(replayable bool, attempts []ProviderAttempt) (string, error) {
    for _, attempt := range attempts {
        if attempt.Err == nil {
            return attempt.Provider, nil
        }
        if attempt.Started || !replayable {
            return "", fmt.Errorf("provider %s failed after output started: %w", attempt.Provider, attempt.Err)
        }
    }
    return "", gatewaylab.ErrNoHealthyProvider
}

In [ ]:
%%
providerTimeout := errors.New("provider timeout")

fallbackProvider, fallbackErr := chooseFallback(true, []ProviderAttempt{
    {Provider: "primary", Err: providerTimeout},
    {Provider: "backup"},
})
fmt.Printf("可重放请求: provider=%s err=%v\n", fallbackProvider, fallbackErr)

_, streamingErr := chooseFallback(true, []ProviderAttempt{
    {Provider: "primary", Started: true, Err: providerTimeout},
    {Provider: "backup"},
})
fmt.Println("已开始流式输出:", streamingErr)

SSE/流式响应一旦写出首字节，再切到 backup 会把两个 provider 的片段混在同一协议流中。此时通常只能中止并记录失败，不能假装无缝重试。

In [ ]:
!*cd "$GO_LEARNING_LABS" && go test ./gateway -run TestRouter -v

## 动手挑战

1. 只对 timeout/5xx fallback，拒绝对 4xx 重试。
2. 加入最大 attempt budget。
3. 设计 sticky route，让同一 session 优先落到同一 provider。

## 复盘

- health、weight 和 policy 谁先参与选择？
- 为什么 fallback 不是简单的 catch？
- 控制面配置验证与数据面请求路由为什么要分开？